# DriftGuard — Data & Model Exploration Notebook

This notebook inspects the raw loan application dataset, the preprocessed 3-way train/validation/test partitions, and evaluates model predictions.

In [ ]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Libraries imported successfully.")

## 1. Raw Dataset Inspection

In [ ]:
raw_path = Path('../dataset/raw/train_u6lujuX_CVtuZ9i.csv')
raw_df = pd.read_csv(raw_path)
print(f"Raw Dataset Shape: {raw_df.shape}")
raw_df.head()

## 2. Target Variable Distribution

In [ ]:
print("Loan Status Distribution:")
print(raw_df['Loan_Status'].value_counts(dropna=False))
print("\nPercentage Distribution:")
print(raw_df['Loan_Status'].value_counts(normalize=True) * 100)

## 3. Preprocessed 3-Way Partitions (Train, Val, Test)

In [ ]:
train_df = pd.read_parquet('../dataset/processed/train.parquet')
val_df = pd.read_parquet('../dataset/processed/val.parquet')
test_df = pd.read_parquet('../dataset/processed/test.parquet')

print(f"Train Partition: {train_df.shape} (70%)")
print(f"Val Partition:   {val_df.shape} (15%)")
print(f"Test Partition:  {test_df.shape} (15%)")

print("\nPreprocessed Train Feature Preview:")
train_df.head()

## 4. Trained XGBoost Classifier Evaluation

In [ ]:
model_path = Path('../models/model.joblib')
model = joblib.load(model_path)

X_test = test_df.drop(columns=['Loan_Status'])
y_test = test_df['Loan_Status'].astype(int)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Model Framework: {model.model_type.upper()}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-Default (0)', 'Default (1)']))